# 15 · Agentic RAG

**Idea:** don't *always* retrieve. Give the model a decision loop: at each step it either **searches** (choosing its own query) or **answers**. This handles questions that need no retrieval, and questions that need several targeted lookups — the model decides.

Our loop is provider-portable: the model emits a small JSON action each step (no native tool-calling required).

> Runs on Ollama. One cell optionally uses Anthropic — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore

#store = NumpyStore(provider='ollama'); store.add(load_markdown())
corpus = load_markdown(directory=str(root / 'multihop'))
print('\n-----SNEHAL START---------------\n')
for co in corpus:
    print(f"Text: {co.text}\n")
    print(f"Source: {co.source}\n")
    print(f"Index: {co.index}\n")
    print(f"Meta: {co.meta}\n")
print('\n-----SNEHAL END---------------\n')
store = NumpyStore(provider='ollama'); store.add(corpus)

A question that needs retrieval. Watch the `searches` trace — the agent chooses what to look up.

In [ ]:
from rag_lab import agentic_answer
from study_buddy import CostTracker

tracker = CostTracker()
# answer, trace = agentic_answer(
#     #'Which embedding model does the service behind the Helpdesk Assistant use?',
#     'What year did the company that acquired the startup founded by Sam\'s college roommate go public?'
#     store, provider='ollama', k=3, max_steps=4, tracker=tracker,
# )
answer, trace = agentic_answer(
    'What year did the company that acquired the startup founded by Sam\'s college roommate go public?',
    store, provider='anthropic', k=5, max_steps=4, tracker=tracker,
)
print('searches:', trace.searches)
print('\nanswer:', answer)
print('\ncost:', tracker.summary())

Now a question that needs **no** retrieval. A good agent answers directly instead of searching — `searches` should stay short or empty.

In [ ]:
answer2, trace2 = agentic_answer('What is 12 times 8?', store, provider='ollama', max_steps=3)
print('searches:', trace2.searches)
print('answer:', answer2)

Optionally drive the same loop with Anthropic. Skips if no key.

In [ ]:
try:
    a, t = agentic_answer(
        'What companies where founded by Sam\'s college roommate?',
        store, provider='anthropic', k=3, max_steps=4,
    )
    print('searches:', t.searches)
    print('answer:', a)
except Exception as e:
    print('anthropic skipped:', type(e).__name__, e)

**Takeaway:** agentic RAG hands *control* of retrieval to the model — it decides whether, what, and how many times to search. More flexible than a fixed pipeline, but each step is a model call, so cap `max_steps`.

Next → `16_graph_rag.ipynb`